# Pipelines en Scikit-Learn

Un **Pipeline** encadena los pasos de preprocesamiento y modelado en un único objeto.

Una de las ventajas es que hace el código más limpio y fácil de reproducir.

In [108]:
### Importar librerias

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')  # Silencia advertencias para mantener el output limpio

# Datos: datasets de ejemplo incluidos en scikit-learn
from sklearn.datasets import load_breast_cancer, load_iris, load_wine, fetch_covtype

# Preprocesamiento: escalar variables numericas e imputar valores faltantes
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# Pipeline y ColumnTransformer: encadenan pasos de preprocesamiento y modelado
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Modelos de clasificacion y regresion
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier

# Herramientas de evaluacion: particion de datos, validacion cruzada y metricas
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import classification_report, mean_squared_error

from sklearn import set_config
set_config(display='diagram')   # Muestra el pipeline como un diagrama visual en Jupyter


## Errores que podríamos evitar si usamos los pipelines

In [6]:
# Cargar el dataset de cancer de mama (clasificacion binaria: maligno vs benigno)
data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)  # Features (30 columnas numericas)
y = data.target  # Etiquetas: 0 = maligno, 1 = benigno

# Dividir en entrenamiento (80%) y prueba (20%) ANTES de escalar
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# FORMA INCORRECTA: el scaler ve TODOS los datos (incluyendo el set de prueba).
# Esto causa 'data leakage': informacion del futuro contamina el entrenamiento.
scaler_malo = StandardScaler()
scaler_malo.fit(X)  # Calcula media/std sobre X completo -> fuga de informacion

# FORMA CORRECTA: el scaler solo aprende del set de entrenamiento.
# El set de prueba se escala con los parametros del entrenamiento, sin ver sus estadisticas.
scaler_bueno = StandardScaler()
scaler_bueno.fit(X_train)  # Solo aprende de X_train -> sin fuga de informacion


StandardScaler()

## ¿Cómo evitar eso con los Pipelines?

In [8]:
# Un Pipeline encadena pasos en orden: primero escala, luego entrena el modelo.
# Esto garantiza que el scaler SIEMPRE se ajuste solo con datos de entrenamiento,
# eliminando el riesgo de data leakage de forma automatica y elegante.
pipe_bueno = Pipeline(
    [
        ('scaler', StandardScaler()),      # Paso 1: estandarizar features (media=0, std=1)
        ('modelo', LogisticRegression())   # Paso 2: clasificador logistico
    ]
)

pipe_bueno  # Mostrar diagrama del pipeline


Pipeline(steps=[('scaler', StandardScaler()), ('modelo', LogisticRegression())])

In [9]:
# .fit() ejecuta todos los pasos en secuencia sobre los datos de entrenamiento:
# 1. StandardScaler aprende media y std de X_train y transforma X_train
# 2. LogisticRegression se entrena con los datos ya escalados
pipe_bueno.fit(X_train, y_train)


Pipeline(steps=[('scaler', StandardScaler()), ('modelo', LogisticRegression())])

In [10]:
# .predict() aplica la misma cadena de transformaciones a los datos nuevos:
# 1. Escala X_test usando los parametros aprendidos de X_train (sin re-ajustar)
# 2. El modelo ya entrenado genera las predicciones
pipe_bueno.predict(X_test)


array([1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1,
       0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1,
       1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1,
       0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0,
       1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1,
       0, 1, 0, 0])

In [11]:
# Reporte de clasificacion: muestra precision, recall y F1-score por clase.
# Permite evaluar que tan bien distingue el modelo entre benigno y maligno.
print(classification_report(y_test, pipe_bueno.predict(X_test)))


              precision    recall  f1-score   support

           0       0.98      0.95      0.96        43
           1       0.97      0.99      0.98        71

    accuracy                           0.97       114
   macro avg       0.97      0.97      0.97       114
weighted avg       0.97      0.97      0.97       114



In [14]:
# Validacion cruzada con 10 folds usando el pipeline completo.
# Importante: al pasar el pipeline a cross_val_score, el scaler se re-ajusta
# en cada fold solo con los datos de entrenamiento de ese fold -> sin data leakage.
cv_score = cross_val_score(pipe_bueno, X, y, cv=10, scoring='accuracy')
print(cv_score.mean())  # Promedio de accuracy en los 10 folds


0.9806704260651629


## Column Transformer



### Iris

In [26]:
# Cargar el dataset Iris: 150 flores con 4 medidas morfologicas
# Clases: 0=Setosa, 1=Versicolor, 2=Virginica
iris = load_iris()


In [27]:
# Crear DataFrame con las medidas de las flores
df = pd.DataFrame(iris.data, columns=iris.feature_names)

# Agregar columna categorica simulada: color de ojos de la flor (inventada)
# Esto es para practicar el manejo de variables categoricas en el pipeline
df['color'] = pd.Categorical(np.random.choice(['cafe', 'miel', 'azul', 'negro', 'verde'], size=len(df)))


In [28]:
# Agregar otra columna categorica simulada: sexo de la planta
# Junto con 'color', servira para demostrar el preprocesamiento de variables categoricas
df['sexo'] = pd.Categorical(np.random.choice(['mujer', 'hombre'], size=len(df)))


In [29]:
df

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),color,sexo
0,5.1,3.5,1.4,0.2,azul,mujer
1,4.9,3.0,1.4,0.2,café,hombre
2,4.7,3.2,1.3,0.2,azul,hombre
3,4.6,3.1,1.5,0.2,negro,hombre
4,5.0,3.6,1.4,0.2,verde,hombre
...,...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,azul,mujer
146,6.3,2.5,5.0,1.9,verde,hombre
147,6.5,3.0,5.2,2.0,azul,mujer
148,6.2,3.4,5.4,2.3,negro,hombre


In [30]:
# Introducir valores faltantes (NaN) en las columnas numericas de forma aleatoria.
# Se eliminan 10 valores por cada columna para simular datos incompletos del mundo real.
rng = np.random.default_rng(0)  # Semilla para reproducibilidad
for col in iris.feature_names:
    idx = rng.choice(df.index, size=10, replace=False)  # Elegir 10 indices al azar
    df.loc[idx, col] = np.nan  # Reemplazar esas filas con NaN


In [31]:
df

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),color,sexo
0,5.1,NaN,NaN,0.2,azul,mujer
1,4.9,3.0,NaN,0.2,café,hombre
2,NaN,3.2,1.3,0.2,azul,hombre
3,4.6,3.1,1.5,0.2,negro,hombre
4,5.0,3.6,NaN,0.2,verde,hombre
...,...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,azul,mujer
146,6.3,2.5,5.0,1.9,verde,hombre
147,6.5,3.0,5.2,2.0,azul,mujer
148,6.2,3.4,5.4,2.3,negro,hombre


In [35]:
# Introducir valores faltantes en las columnas categoricas.
# Se eliminan 8 valores en 'color' y 'sexo' para simular datos sucios.
rng = np.random.default_rng(0)
for col in ['color', 'sexo']:
    idx = rng.choice(df.index, size=8, replace=False)
    df.loc[idx, col] = np.nan


In [36]:
df

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),color,sexo
0,5.1,NaN,NaN,0.2,azul,NaN
1,4.9,3.0,NaN,0.2,café,hombre
2,NaN,3.2,1.3,0.2,NaN,hombre
3,4.6,3.1,1.5,0.2,negro,hombre
4,5.0,3.6,NaN,0.2,verde,hombre
...,...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,azul,mujer
146,6.3,2.5,5.0,1.9,verde,hombre
147,6.5,3.0,5.2,2.0,azul,mujer
148,6.2,3.4,5.4,2.3,negro,hombre


In [32]:
# Separar las columnas segun su tipo para aplicar transformaciones distintas:
# - Numericas: imputar con mediana + estandarizar
# - Categoricas: imputar con moda + codificar con One-Hot Encoding
columnas_numericas = ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
columnas_catogericas = ['color', 'sexo']


In [48]:
# Pipeline para columnas NUMERICAS:
# 1. Imputer: rellena NaN con la mediana de cada columna (robusta a outliers)
# 2. Scaler: estandariza a media=0, std=1 para que el modelo no se sesgue por escala
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Pipeline para columnas CATEGORICAS:
# 1. Imputer: rellena NaN con la categoria mas frecuente (moda)
# 2. OneHotEncoder: convierte cada categoria en una columna binaria (0 o 1)
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder())
])

# ColumnTransformer: aplica cada pipeline al subconjunto correcto de columnas.
# Las columnas no mencionadas se descartan por defecto (remainder='drop').
preprocesador = ColumnTransformer([
    ('num', num_pipeline, columnas_numericas),   # Aplica num_pipeline a columnas numericas
    ('cat', cat_pipeline, columnas_catogericas)  # Aplica cat_pipeline a columnas categoricas
])

# Pipeline final: preprocesamiento + modelo de clasificacion en un solo objeto
pipe_final = Pipeline([
    ('preprocesador', preprocesador),
    ('modelo', LogisticRegression())
])

pipe_final  # Mostrar diagrama completo del pipeline


Pipeline(steps=[('preprocesador',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['sepal length (cm)',
                                                   'sepal width (cm)',
                                                   'petal length (cm)',
                                                   'petal width (cm)']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder())]),
                                                  ['color', 'sexo'])])),
                ('modelo', LogisticRegression())])

In [49]:
# X son todas las columnas del DataFrame (features numericas y categoricas con NaN)
X = df
# y son las etiquetas reales del dataset Iris (0=Setosa, 1=Versicolor, 2=Virginica)
y = iris.target


In [50]:
y

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2])

In [51]:
# Dividir datos en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# El pipeline ajusta el preprocesador y el modelo solo con X_train
pipe_final.fit(X_train, y_train)

# Predecir sobre X_test: el pipeline transforma X_test con los parametros de X_train
y_pred = pipe_final.predict(X_test)

# Comparar predicciones contra etiquetas reales
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      1.00      1.00         9
           2       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30



### Vino

In [58]:
# Cargar dataset Vino: 178 muestras de vino con 13 propiedades quimicas
# Objetivo: clasificar el vino en una de 3 variedades
wine = load_wine()
df = pd.DataFrame(wine.data, columns=wine.feature_names)

# Agregar columna categorica simulada: pais de origen (inventada para el ejercicio)
df['pais'] = pd.Categorical(np.random.choice(['Chile', 'Francia', 'Colombia', 'Argentina'], size=len(df)))
df


,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,pais
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,Colombia
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,Argentina
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,Argentina
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,Colombia
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,Colombia
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173,13.71,5.65,2.45,20.5,95.0,1.68,0.61,0.52,1.06,7.70,0.64,1.74,740.0,Francia
174,13.40,3.91,2.48,23.0,102.0,1.80,0.75,0.43,1.41,7.30,0.70,1.56,750.0,Argentina
175,13.27,4.28,2.26,20.0,120.0,1.59,0.69,0.43,1.35,10.20,0.59,1.56,835.0,Argentina
176,13.17,2.59,2.37,20.0,120.0,1.65,0.68,0.53,1.46,9.30,0.60,1.62,840.0,Francia


In [62]:
# Definir que columnas son numericas y cuales categoricas para este dataset
columnas_numericas = wine.feature_names  # Las 13 propiedades quimicas del vino
columnas_catogericas = ['pais']           # La columna de pais que agregamos


In [59]:
# Simular datos faltantes en la columna categorica 'pais':
# Se eliminan 8 registros de forma aleatoria
rng = np.random.default_rng(0)
for col in ['pais']:
    idx = rng.choice(df.index, size=8, replace=False)
    df.loc[idx, col] = np.nan
df


,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,pais
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,Colombia
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,Argentina
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,NaN
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,Colombia
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,Colombia
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173,13.71,5.65,2.45,20.5,95.0,1.68,0.61,0.52,1.06,7.70,0.64,1.74,740.0,Francia
174,13.40,3.91,2.48,23.0,102.0,1.80,0.75,0.43,1.41,7.30,0.70,1.56,750.0,Argentina
175,13.27,4.28,2.26,20.0,120.0,1.59,0.69,0.43,1.35,10.20,0.59,1.56,835.0,Argentina
176,13.17,2.59,2.37,20.0,120.0,1.65,0.68,0.53,1.46,9.30,0.60,1.62,840.0,Francia


In [64]:
# Simular datos faltantes en las columnas numericas:
# Se eliminan 15 valores por cada una de las 13 propiedades quimicas
rng = np.random.default_rng(0)
for col in columnas_numericas:
    idx = rng.choice(df.index, size=15, replace=False)
    df.loc[idx, col] = np.nan
df


,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,pais
0,14.23,NaN,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,Colombia
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,NaN,NaN,1050.0,Argentina
2,NaN,2.36,2.67,18.6,101.0,NaN,NaN,0.30,2.81,5.68,1.03,3.17,NaN,NaN
3,14.37,NaN,NaN,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,Colombia
4,13.24,NaN,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,Colombia
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173,13.71,5.65,2.45,20.5,95.0,NaN,NaN,0.52,1.06,7.70,0.64,1.74,740.0,Francia
174,13.40,3.91,2.48,NaN,102.0,1.80,0.75,0.43,1.41,7.30,0.70,NaN,750.0,Argentina
175,13.27,4.28,2.26,20.0,120.0,1.59,0.69,0.43,NaN,10.20,NaN,1.56,835.0,Argentina
176,13.17,2.59,2.37,20.0,120.0,1.65,0.68,0.53,NaN,9.30,0.60,1.62,840.0,Francia


In [65]:
# Funcion reutilizable que construye el pipeline completo para datos mixtos.
# Recibe las listas de columnas y devuelve un Pipeline listo para usar.
# Ventaja: se puede llamar con distintos datasets sin repetir codigo.
def crear_pipeline_mixto(columnas_numericas, columnas_catogericas):
    # Pipeline para columnas numericas: imputa con mediana y estandariza
    num_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    # Pipeline para columnas categoricas: imputa con moda y aplica One-Hot Encoding
    cat_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder())
    ])

    # ColumnTransformer: dirige cada pipeline al tipo de columna correspondiente
    preprocesador = ColumnTransformer([
        ('num', num_pipeline, columnas_numericas),
        ('cat', cat_pipeline, columnas_catogericas)
    ])

    # Pipeline final: preprocesamiento + modelo de clasificacion
    pipe_final = Pipeline([
        ('preprocesador', preprocesador),
        ('modelo', LogisticRegression())
    ])

    return pipe_final


In [68]:
# Crear el pipeline para el dataset de vino usando la funcion reutilizable
pipe_vino = crear_pipeline_mixto(columnas_numericas, columnas_catogericas)
pipe_vino  # Mostrar diagrama del pipeline


Pipeline(steps=[('preprocesador',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['alcohol', 'malic_acid',
                                                   'ash', 'alcalinity_of_ash',
                                                   'magnesium', 'total_phenols',
                                                   'flavanoids',
                                                   'nonflavanoid_phenols',
                                                   'proanthocyanins',
                                                   'color_intensity', 'hue',
                                                   'od280/od315_of_diluted_wines',
                                                   'proline']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder())]),
                                                  ['pais'])])),
                ('modelo', LogisticRegression())])

In [70]:
# Dividir el dataset de vino en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(df, wine.target, test_size=0.2, random_state=42)

# Ajustar el pipeline: imputa, escala y entrena solo con datos de entrenamiento
pipe_vino.fit(X_train, y_train)

# Predecir la variedad de vino para el set de prueba
y_pred = pipe_vino.predict(X_test)

# Evaluar el desempeno del modelo
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.93      1.00      0.97        14
           1       1.00      0.86      0.92        14
           2       0.89      1.00      0.94         8

    accuracy                           0.94        36
   macro avg       0.94      0.95      0.94        36
weighted avg       0.95      0.94      0.94        36



# Una breve introducción de cómo es la Programación Orientada a Objetos en Python

POO (OOP en inglés) organiza el código alrededor de clases.

#### ¿Qué son las clases?
Un ejemplo sencillo

In [88]:
# VERSION PROBLEMATICA de la clase Animal:
# diccionario_animales y contador son atributos de CLASE (compartidos entre instancias).
# Problema: el contador no se incrementa correctamente porque en Python los enteros
# son inmutables; 'self.contador += 1' crea un atributo de instancia, no modifica el de clase.
class Animal():

    diccionario_animales = {}  # Compartido entre TODAS las instancias (atributo de clase)
    contador = 1               # Tambien de clase, pero su incremento tiene comportamiento inesperado

    # Constructor: se ejecuta automaticamente al crear una instancia
    def __init__(self, nombre, edad):
        self.nombre = nombre  # Atributo de instancia: unico para cada animal
        self.edad = edad
        self.diccionario_animales[self.contador] = self.nombre  # Registrar en el diccionario
        self.contador += 1  # Crea un 'contador' local a la instancia, no modifica el de clase

        print("Hola, soy " + nombre + " y tengo " + str(edad) + " anos")

    def comer(self):
        print("Estoy comiendo")

    def dormir(self):
        print("ZZZZ")

    def respirar(self):
        print("Inhala...Exhala")


In [94]:
# Solucion: usar variables GLOBALES para que persistan correctamente entre instancias.
# En Python, las variables globales viven en el modulo y cualquier funcion/clase puede modificarlas
# siempre que se declaren con 'global' dentro del metodo.
diccionario_animales_global = {}  # Diccionario que registrara todos los animales creados
contador_global = 1               # Contador que se incrementara correctamente con cada nuevo animal


In [95]:
# VERSION CORREGIDA de la clase Animal usando variables globales.
# Al declarar 'global contador_global' dentro de __init__, Python entiende que queremos
# modificar la variable del modulo, no crear una local.
class Animal():

    # Constructor: inicializa cada animal y lo registra en el diccionario global
    def __init__(self, nombre, edad):
        self.nombre = nombre  # Atributo propio del animal
        self.edad = edad
        global diccionario_animales_global  # Indica que usamos la variable del modulo
        global contador_global
        diccionario_animales_global[contador_global] = self.nombre  # Guardar con indice unico
        contador_global += 1  # Incrementa correctamente la variable global

        print("Hola, soy " + nombre + " y tengo " + str(edad) + " anos")

    def comer(self):
        print("Estoy comiendo")

    def dormir(self):
        print("ZZZZ")

    def respirar(self):
        print("Inhala...Exhala")


In [96]:
# Crear instancias de Animal: cada llamada ejecuta __init__ automaticamente
pepe = Animal("Pepe", 5)          # Se registra en el diccionario con indice 1
pepe.comer()                       # Llamar al metodo de instancia
print("Diccionario despues de Pepe:", diccionario_animales_global)

trufa = Animal("Trufa", 7)         # Se registra con indice 2
trufa.comer()
print("Diccionario despues de Trufa:", diccionario_animales_global)

print("Diccionario final:", diccionario_animales_global)  # Muestra todos los animales creados


Hola, soy Pepe y tengo 5 años
Estoy comiendo
Diccionario después de Pepe: {1: 'Pepe'}
Hola, soy Trufa y tengo 7 años
Estoy comiendo
Diccionario después de Trufa: {1: 'Pepe', 2: 'Trufa'}
Diccionario final: {1: 'Pepe', 2: 'Trufa'}


#### Herencia

In [97]:
# Herencia: Perro extiende Animal heredando todos sus metodos (comer, dormir, respirar).
# super().__init__() llama al constructor del padre para inicializar nombre y edad.
# Perro agrega su propio atributo: 'raza'.
class Perro(Animal):
    def __init__(self, nombre, edad, raza):
        super().__init__(nombre, edad)  # Ejecuta el __init__ de Animal (registra en diccionario)
        self.raza = raza                # Atributo exclusivo de la clase Perro
        print("Soy un perro de raza " + raza)


In [99]:
# Crear un Perro: hereda el registro en el diccionario global de Animal
# y ademas guarda su raza
firulais = Perro("Firulais", 3, "Labrador")
firulais.comer()  # Metodo heredado de Animal, funciona sin redefinirlo


Hola, soy Firulais y tengo 3 años
Soy un perro de raza Labrador
Estoy comiendo


In [100]:
# Verificar que Firulais tambien quedo registrado en el diccionario global
# (heredo el comportamiento del __init__ de Animal)
diccionario_animales_global


{1: 'Pepe', 2: 'Trufa', 3: 'Firulais', 4: 'Firulais'}

#### Aplicación a Data Science

In [106]:
# Cargar el dataset de alturas desde un archivo Excel
# Contiene medidas corporales para predecir la altura de una persona
alturas = pd.read_excel('alturas.xlsx')
alturas.head()  # Ver las primeras 5 filas para conocer la estructura


,Peso,Mujer,CaloriasAlDia,AlturaMama,AlturaPapa,AlturaPersona
0,70,0,2726,145,160,158
1,62,0,2072,158,160,160
2,70,0,2994,166,160,167
3,90,0,2923,172,160,160
4,82,0,2620,147,161,162


In [107]:
# Clase que encapsula todo el flujo de una regresion lineal:
# creacion del modelo, entrenamiento y evaluacion.
# Ventaja de POO: el estado (X, y, modelo, predicciones) vive en el objeto,
# sin necesidad de pasar variables entre funciones.
class RegresionCompleta:

    # Constructor: guarda los datos como atributos del objeto
    def __init__(self, X, y):
        self.X = X  # Features (variables independientes)
        self.y = y  # Variable objetivo

    def crear_modelo(self):
        # Instanciar el modelo de regresion lineal y guardarlo como atributo
        # para que otros metodos puedan acceder a el
        self.modelo = LinearRegression()

    def evaluar(self):
        # Dividir datos, entrenar y evaluar en una sola operacion
        X_train, X_test, y_train, y_test = train_test_split(
            self.X, self.y, test_size=0.2, random_state=42
        )
        self.modelo.fit(X_train, y_train)           # Ajustar el modelo con datos de entrenamiento
        self.y_pred = self.modelo.predict(X_test)   # Generar predicciones sobre datos de prueba

        # RMSE: raiz del error cuadratico medio, en las mismas unidades que y (cm de altura)
        # Cuanto mas bajo, mejor predice el modelo
        rmse = np.sqrt(mean_squared_error(y_test, self.y_pred))
        print('RMSE:', rmse)


In [116]:
# Separar features y variable objetivo
X = alturas.drop(['AlturaPersona'], axis=1)  # Todo excepto la columna a predecir
y = alturas['AlturaPersona']                 # Lo que queremos predecir

# Crear una instancia de RegresionCompleta con los datos de alturas
regresion_alturas = RegresionCompleta(X, y)

# Crear el modelo (instancia LinearRegression internamente)
regresion_alturas.crear_modelo()

# Entrenar y evaluar: imprime el RMSE del modelo sobre el set de prueba
regresion_alturas.evaluar()


RMSE: 5.362913349316423
